## Create binary MMSE_Category Label, and new delta table

- Look at comparing NC/Mild, NC/Mod, NC/Sev, NC/Pos-MCI

In [0]:
%sql
CREATE OR REPLACE TABLE multimodal_adni_living_binary AS
  SELECT *,
    CASE 
      WHEN MMSE_Category IN (
        'Mild AD',
        'Moderate AD',
        'Severe AD'
      ) THEN 1
      WHEN MMSE_Category IN (
        'Normal cognition (Fully intact)',
        'Normal cognition / Possible MCI (Borderline)'
      ) THEN 0
      WHEN MMSE_Category = 'Invalid Score' THEN NULL  
      ELSE NULL 
  END AS y_mmse_binary
FROM final_multimodal_adni_living

## Remove Data Leakage & PET Columns
Removing any columns that might cause a Data Leakage (ID columns), as well as any PET Columns, since we arent using any image data, and just want biomarkers (probably).

### Pull delta table into spark and check class imbalance
Checking how many positive vs negative cases.

In [0]:
from pyspark.sql import functions as F
table_name = "multimodal_adni_living_binary"

df = spark.table(table_name)

# remove rows where label is null/missing
df_model = df.filter(F.col("y_mmse_binary").isNotNull())

# drop data leakage/IDs
id_drop_cols = ["PTID", "MMSE_Category", "image_id", "study_id", "MMSCORE", "WORLDSCORE", "NPISCORE"] # ID Columns, but ALSO MMScore is leakage because it directly makes the MMSE_Category variable, causing it to also be a data leakage.
df_model = df_model.drop(*[c for c in id_drop_cols if c in df_model.columns]) # drop columns if they exist in the dataframe

# drop 100% missing column VSHGTSC
df_model = df_model.drop("VSHGTSC")

# drop pet stuff
pet_drop_cols = [
    "pet_visit",
    "pet_date",
    "pet_description",
    "pet_mfr",
    "pet_mfr_model",
    "pet_reconstruct",
    "pet_height",
    "pet_width",
    "pet_n_images",
    "pet_n_frames",
    "pet_pixel_x",
    "pet_pixel_y",
    "pet_thickness",
    "pet_cnts_src",
    "pet_radiopharm",
    "pet_isotope",
    "pet_processing",
    "pet_decay_corr",
    "pet_sctr_corr",
    "pet_atten_corr",
    "pet_rndm_corr",
    "pet_cv_kernel",
    "image_type", #2/27 PET type column, doesn't add anything to the model
    "DD_CRF_VERSION_LABEL", # PET type column, doesn't add anything, same value for all 266 pet-included records
    "LANGUAGE_CODE", # Language used, all are either NULL or E (assuming english)
    "HAS_QC_ERROR", # all are null or 0, for no error
]

df_model = df_model.drop(*[c for c in pet_drop_cols if c in df_model.columns]) # again, check above comment


Check count of positive and negative cases

In [0]:
display(df_model.select("y_mmse_binary").groupBy("y_mmse_binary").count())

Variable Check

In [0]:
for col in df_model.columns:
    print(col)

Write the dataframe as a Delta Table

In [0]:
df_model.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("multimodal_adni_living_model_ready")

View Delta Table

In [0]:
%sql
SELECT *
FROM multimodal_adni_living_model_ready
LIMIT 10;

In [0]:
%sql
ALTER TABLE multimodal_adni_living_model_ready SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');
ALTER TABLE multimodal_adni_living_model_ready
DROP COLUMNS DD_CRF_VERSION_LABEL, LANGUAGE_CODE, HAS_QC_ERROR, image_type;

In [0]:
spark.sql("DESCRIBE DETAIL multimodal_adni_living_model_ready").display()